<div class="alert alert-block alert-danger">
Before you start running this notebook, create a Hail-enabled Dataproc cluster with a <b>high compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab Spark cluster</b> and, under <b>Software to install</b>, choose <b>Hail</b>.</li>
    <li>Configure a high-compute cluster (multiple workers). Cluster creation can take about 10 minutes.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a> and <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/cloud_app_types/dataproc/">Use Dataproc (Spark) and Hail</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook compiles variant-class counts for a Phase 1 inventory table. It intersects short-read and long-read sample lists, subsets short-read SV and Hail VDS/MatrixTable callsets, and tallies SNVs, indels, and per-sample SVTYPEs.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
# List the Controlled Tier v7 WGS data and the workspace bucket.
!gsutil -u $GOOGLE_PROJECT ls gs://fc-aou-datasets-controlled/v7/wgs

In [ ]:
!gsutil -u $GOOGLE_PROJECT ls $WORKSPACE_BUCKET/

In [ ]:
import pandas as pd
import numpy as np
import pysam
import os

In [ ]:
import pysam
from pysam import VariantFile

from collections import defaultdict
from collections import Counter

from tqdm.notebook import tqdm

In [ ]:
import hail as hl
from hail.plot import show
from pprint import pprint

In [ ]:
# Workspace details the Workbench sets as environment variables.
bucket = os.environ['WORKSPACE_BUCKET']
workspace = os.environ['WORKSPACE_NAME']
namespace = os.environ['WORKSPACE_NAMESPACE']

In [ ]:
# Controlled Tier short-read SV callset (GATK-SV), downloaded once (requester-pays bucket).
if not os.path.exists("AoU_srWGS_SV.vcf.gz"):
    !gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v7/wgs/short_read/structural_variants/vcf/AoU_srWGS_SV.vcf.gz .
    !gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v7/wgs/short_read/structural_variants/vcf/AoU_srWGS_SV.vcf.gz.tbi .

In [ ]:
# Sample IDs in the short-read SV VCF header.
sr_sv_samples = !zgrep -m1 '^#CHROM' AoU_srWGS_SV.vcf.gz | cut -f10- | sed 's/\t/\n/g'

In [ ]:
len(sr_sv_samples)

In [ ]:
# Sample IDs in the long-read joint-called VCF header (the 1,027-sample long-read cohort).
lr_sv_samples = !gsutil -u $GOOGLE_PROJECT cat gs://fc-aou-datasets-controlled/v7/wgs/long_read/joint_vcf/GRCh38/cohort_for_GLNexus_2023Q1_1027.g.vcf.bgz | zgrep -m1 '^#CHROM' | cut -f10- | sed 's/\t/\n/g'
len(lr_sv_samples)

In [ ]:
# Cohorts: all 1,027 long-read samples, and the 989 of them that also have short-read SV calls.
common_samples_1027 = lr_sv_samples
len(common_samples_1027)

In [ ]:
common_samples_989 = list(set(sr_sv_samples) & set(lr_sv_samples))
len(common_samples_989)

In [ ]:
# Start Hail with more driver and executor memory.
spark_conf_more_ram = dict()
spark_conf_more_ram["spark.executor.memory"] = "8g"
spark_conf_more_ram["spark.driver.memory"] = "196g"

# hl.init(default_reference='GRCh38', idempotent=True, spark_conf=spark_conf_more_ram)

hl.init(idempotent=True, spark_conf=spark_conf_more_ram)

In [ ]:
hl.default_reference('GRCh38')

In [ ]:
# The Controlled Tier short-read SNV/indel callset (VDS). Subset it to the long-read samples and save.
vds = hl.vds.read_vds('gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/vds/hail.vds')

In [ ]:
callset_sample_filtered_1027 = hl.vds.filter_samples(vds, common_samples_1027, keep=True, remove_dead_alleles=True)

In [ ]:
callset_sample_filtered_1027.write(f'{bucket}/scratch/srs-subset.1027.vds', overwrite=True)

In [ ]:
callset_sample_filtered_1027 = hl.vds.read_vds(f'{bucket}/scratch/srs-subset.1027.vds')

In [ ]:
common_samples_1027

In [ ]:
# Exploratory: list available short-read callsets, including the older v6 MatrixTable.
!gsutil -u $GOOGLE_PROJECT ls gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/vds/

In [ ]:
# The v6 short-read MatrixTable, subset to the long-read samples and saved.
mt = hl.read_matrix_table('gs://fc-aou-datasets-controlled/v6/wgs/hail.mt')

In [ ]:
len(mt.s.collect())

In [ ]:
mt_subset = mt.filter_cols(hl.set(common_samples_1027).contains(mt.s))

In [ ]:
mt_subset.write(f'{bucket}/scratch/srs-subset.1027.mt', overwrite=True)

In [ ]:
mt_subset = hl.read_matrix_table(f'{bucket}/scratch/srs-subset.1027.mt')

In [ ]:
sample_names = mt_subset.s.collect()

In [ ]:
sample_names

In [ ]:
# Exploratory: list the v5 MatrixTable, the environment (WDS_VDS_PATH), and one exome VCF's samples.
!gsutil -u $GOOGLE_PROJECT ls gs://fc-aou-datasets-controlled/v5/wgs/hail.mt/

In [ ]:
!set

In [ ]:
!gsutil -u $GOOGLE_PROJECT ls $WDS_VDS_PATH

In [ ]:
!gsutil -u $GOOGLE_PROJECT cat gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/exome_v7.1/vcf/exome.chr22.vcf.bgz | zgrep -m1 '^#CHROM' | cut -f10- | sed 's/\t/\n/g'
    

In [ ]:
!gsutil -u $GOOGLE_PROJECT ls gs://fc-aou-datasets-controlled/v7/wgs/short_read/structural_variants/vcf/
        

In [ ]:
# Write the 989 shared sample IDs, one per line, for bcftools.
with open('sample_list.txt', 'w') as sample_list:
    for sample_name in common_samples_989:
        sample_list.write(f'{sample_name}\n')

In [ ]:
!cat sample_list.txt

In [ ]:
# Subset the short-read SV VCF to the 989 shared samples.
!bcftools

In [ ]:
!bcftools view -S sample_list.txt --force-samples -o AoU_srWGS_SV.subset.vcf.gz -O z AoU_srWGS_SV.vcf.gz

In [ ]:
!ls -lh

In [ ]:
# Count, for each sample, the SVs of each SVTYPE where the sample carries a non-reference allele.
def count_svtype_per_sample(vcf_file_path):
    """
    Iterates over each line in a VCF file and counts the number of variants
    of each SVTYPE for each sample.

    Parameters:
        vcf_file_path (str): Path to the VCF file.

    Returns:
        dict: A nested dictionary where the outer keys are sample names,
              inner keys are SVTYPEs, and values are counts.
    """
    # Open the VCF file
    try:
        vcf = pysam.VariantFile(vcf_file_path)
    except FileNotFoundError:
        raise FileNotFoundError(f"VCF file not found at: {vcf_file_path}")
        
    num_records_line = !zgrep -vc ^'#' {vcf_file_path}
    num_records = int(num_records_line[0])
    
    # Initialize the dictionary for counts
    svtype_counts = defaultdict(lambda: defaultdict(int))
    
    # Iterate through each record in the VCF
    for record in tqdm(vcf.fetch(), total=num_records, desc="Processing VCF records"):
        # Get the SVTYPE from the INFO field
        svtype = record.info.get('SVTYPE', None)
        if not svtype:
            continue  # Skip if SVTYPE is not present
        
        # Update counts for each sample
        for sample in record.samples:
            genotype = record.samples[sample].get('GT')
            if genotype is None or any(gt is None for gt in genotype):
                continue  # Skip no-calls (./.)
            
            if any(gt != 0 for gt in genotype):  # Check for non-reference alleles
                svtype_counts[sample][svtype] += 1
                
    return svtype_counts

In [ ]:
# Per-sample SV counts in the short-read callset, saved as JSON.
sv_short_reads_counts = count_svtype_per_sample('AoU_srWGS_SV.subset.vcf.gz')

In [ ]:
sv_short_reads_counts

In [ ]:
import json

with open('sv_sens_sr_counts.json', 'w') as json_file:
    json.dump(sv_short_reads_counts, json_file, indent=4)

In [ ]:
# Stacked bar plot of per-sample insertions and unknown-type SVs (above zero) and deletions (below zero).
import matplotlib.pyplot as plt
import numpy as np

def plot_sv_counts(svtype_counts):
    """
    Creates a stacked bar plot of SV counts for each sample.

    Parameters:
        svtype_counts (dict): Nested dictionary with sample names as keys, 
                              SVTYPEs as subkeys, and counts as values.
    """
    # Extract sample names and SVTYPEs
    samples = list(svtype_counts.keys())
    svtypes = ['INS', 'DEL', 'UNK']  # Define SV types to include in the plot

    # Prepare data for insertions and deletions
    insertion_counts = [svtype_counts[sample].get('INS', 0) for sample in samples]
    deletion_counts = [svtype_counts[sample].get('DEL', 0) for sample in samples]
    unknown_counts = [svtype_counts[sample].get('UNK', 0) for sample in samples]

    # Define x-axis positions for samples
    x = np.arange(len(samples))

    # Set up the figure
    fig, ax = plt.subplots(figsize=(12, 8))

    # Stack insertions and unknowns for positive y-axis
    ax.bar(x, insertion_counts, label='Insertions', color='skyblue')
    ax.bar(x, unknown_counts, bottom=insertion_counts, label='Unknowns', color='lightgreen')

    # Stack deletions for negative y-axis
    ax.bar(x, [-count for count in deletion_counts], label='Deletions', color='salmon')

    # Customize the plot
    ax.axhline(0, color='black', linewidth=0.8)  # Add a horizontal line at y=0
    ax.set_xticks(x)
    ax.set_xticklabels(samples, rotation=45, ha='right')  # Rotate x-axis labels for better visibility
    ax.set_ylabel('SV Counts')
    ax.set_title('Stacked Bar Plot of SV Counts')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))  # Place legend outside the plot

    # Adjust layout
    plt.tight_layout()
    plt.show()

In [ ]:
plot_sv_counts(sv_short_reads_counts)

In [ ]:
# Variant of the plot above that stacks three callsets per sample; called here with made-up example values.
import matplotlib.pyplot as plt
import numpy as np

def plot_sv_counts_multiple(dict1, dict2, dict3):
    """
    Creates a stacked bar plot of SV counts for each sample using three SV count dictionaries.

    Parameters:
        dict1, dict2, dict3 (dict): Nested dictionaries with sample names as keys,
                                    SVTYPEs as subkeys, and counts as values.
                                    Sample names must be the same across all dictionaries.
    """
    # Extract sample names (assume same order for all dictionaries)
    samples = list(dict1.keys())
    svtypes = ['INS', 'DEL', 'UNK']  # Define SV types to include in the plot

    # Prepare data arrays for each SVTYPE and dictionary
    data = {svtype: {'dict1': [], 'dict2': [], 'dict3': []} for svtype in svtypes}
    for sample in samples:
        for svtype in svtypes:
            data[svtype]['dict1'].append(dict1[sample].get(svtype, 0))
            data[svtype]['dict2'].append(dict2[sample].get(svtype, 0))
            data[svtype]['dict3'].append(dict3[sample].get(svtype, 0))

    # Define x-axis positions for samples
    x = np.arange(len(samples))

    # Set up the figure
    fig, ax = plt.subplots(figsize=(14, 8))

    # Initialize bottom arrays for positive (insertions/unknowns) and negative (deletions) stacks
    bottom_insertion = np.zeros(len(samples))
    bottom_deletion = np.zeros(len(samples))

    # Stack for each SVTYPE and dictionary
    for svtype in ['INS', 'UNK']:  # Positive y-axis
        for label, color, counts in zip(['dict1', 'dict2', 'dict3'], 
                                        ['skyblue', 'cornflowerblue', 'lightblue'], 
                                        [data[svtype]['dict1'], data[svtype]['dict2'], data[svtype]['dict3']]):
            ax.bar(x, counts, bottom=bottom_insertion, label=f'{svtype} ({label})', color=color)
            bottom_insertion += counts

    for svtype in ['DEL']:  # Negative y-axis
        for label, color, counts in zip(['dict1', 'dict2', 'dict3'], 
                                        ['salmon', 'lightcoral', 'darksalmon'], 
                                        [data[svtype]['dict1'], data[svtype]['dict2'], data[svtype]['dict3']]):
            ax.bar(x, [-count for count in counts], bottom=bottom_deletion, label=f'{svtype} ({label})', color=color)
            bottom_deletion -= counts

    # Customize the plot
    ax.axhline(0, color='black', linewidth=0.8)  # Add a horizontal line at y=0
    ax.set_xticks(x)
    ax.set_xticklabels(samples, rotation=45, ha='right')  # Rotate x-axis labels for better visibility
    ax.set_ylabel('SV Counts')
    ax.set_title('Stacked Bar Plot of SV Counts for Each Sample')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))  # Place legend outside the plot

    # Adjust layout
    plt.tight_layout()
    plt.show()

# Example usage
dict1 = {
    'SAMPLE1': {'INS': 15000, 'DEL': 10000, 'UNK': 30},
    'SAMPLE2': {'INS': 16000, 'DEL': 11000, 'UNK': 30},
    'SAMPLE3': {'INS': 16500, 'DEL': 10500, 'UNK': 35},
}
dict2 = {
    'SAMPLE1': {'INS': 12000, 'DEL': 9000, 'UNK': 20},
    'SAMPLE2': {'INS': 13000, 'DEL': 9200, 'UNK': 25},
    'SAMPLE3': {'INS': 13500, 'DEL': 9100, 'UNK': 20},
}
dict3 = {
    'SAMPLE1': {'INS': 14000, 'DEL': 9500, 'UNK': 30},
    'SAMPLE2': {'INS': 14500, 'DEL': 9600, 'UNK': 30},
    'SAMPLE3': {'INS': 15000, 'DEL': 9400, 'UNK': 30},
}

plot_sv_counts_multiple(dict1, dict2, dict3)